In [ ]:
import sys
sys.path.append("<PROJECT_ROOT>")
# Replace this placeholder with your local project root if needed.
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from dataset import SeismicDataset
from net.Diffusion import (ConditionalDiffusionUNet, SubVPDiffusionManager, 
                           DDIMSampler)
import pandas as pd
DATA_ROOT  = "<DATASET_ROOT>"
# Replace this placeholder with your local dataset root directory.
CSV_PATH   = os.path.join(DATA_ROOT, "<DATASET_ANNOTATION_CSV>")
# Replace this placeholder with your dataset annotation CSV file name.
MODEL_PATH = 'experiments/run_disp012/checkpoints/best_model.pth'
DATA_TYPE  = 'disp'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
N_ENSEMBLE = 10

train_ds = SeismicDataset(DATA_ROOT, CSV_PATH, data_type=DATA_TYPE,
                          layer_combo=[0,1,2], split='train')
val_ds   = SeismicDataset(DATA_ROOT, CSV_PATH, data_type=DATA_TYPE,
                          layer_combo=[0,1,2], split='val',
                          external_scale=train_ds.scale)

model = ConditionalDiffusionUNet(
    ground_channels=2,
    condition_channels=val_ds.condition_channels
).to(DEVICE)
model.load_state_dict(
    torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
)
model.eval()

dm      = SubVPDiffusionManager(timesteps=1000, device=DEVICE)
sampler = DDIMSampler(dm, ddim_steps=50)

print(f"Model loaded | Val samples: {len(val_ds)} | Scale: {val_ds.scale:.4f}")

In [ ]:
csv_path = 'experiments/run_disp012/full_record_results/metrics_summary.csv'
df = pd.read_csv(csv_path)
df['corr_avg'] = (df['corr_ns'] + df['corr_ew']) / 2
# df['eSa_avg']  = (df['eSa_ns_%'] + df['eSa_ew_%']) / 2

df_sorted = df.sort_values('corr_avg', ascending=False)
print(df_sorted[['filename','corr_ns','corr_ew','corr_avg']].head(8))

target_fname = df_sorted.iloc[0]['filename'].replace('.npz','')
print(f"\nSelected: {target_fname}")
print(f"CorrNS={df_sorted.iloc[0]['corr_ns']:.3f}  "
      f"CorrEW={df_sorted.iloc[0]['corr_ew']:.3f}  ")
      # f"eSa_avg={df_sorted.iloc[0]['eSa_avg']:.1f}%")

In [ ]:

target_fname = 'fzcgznRun086'
print(f"Selected: {target_fname}")
idx = 0
target_indices = []
for meta in val_ds.target_files:
    fname = meta['filename'].replace('.npz', '')
    npz_path = os.path.join(val_ds.root_dir, meta['folder'], fname + '.npz')
    try:
        data      = np.load(npz_path, allow_pickle=True)
        total_len = data['labels'].shape[0]
        n_win     = max(1, (total_len - 1024) // 512 + 1)
    except:
        n_win = 1
    if fname == target_fname:
        target_indices = list(range(idx, idx + n_win))
        break
    idx += n_win

print(f"Found {len(target_indices)} windows for {target_fname}")

max_energy = -1
best_idx   = target_indices[0]
for i in target_indices:
    s      = val_ds[i]
    # The ground motion is already normalized.
    energy = s['ground'].abs().max().item()
    if energy > max_energy:
        max_energy = energy
        best_idx   = i

print(f"Best window index: {best_idx}  (max energy: {max_energy:.4f})")

sample    = val_ds[best_idx]
condition = sample['layers'].unsqueeze(0).to(DEVICE)
gt        = sample['ground'].numpy() * val_ds.scale   # [2, 1024]
print(f"GT max: {np.abs(gt).max():.4f}  (should be non-zero)")

In [ ]:
from reconstruct_full import invert_record, detect_key_segment

@torch.no_grad()
def ensemble_uq_full(model, sampler, record_samples, device, 
                     scale, n=10, window_size=1024, stride=512):
    """
    Run N Hann-OLA reconstructions for a full record and return the mean and standard deviation time histories.
    """
    all_preds = []
    for _ in range(n):
        pred_windows, gt_windows = [], []
        for sample in record_samples:
            layers_raw = sample['layers']
            ground_raw = sample['ground']
            if not isinstance(layers_raw, torch.Tensor):
                layers_raw = torch.from_numpy(layers_raw.astype(np.float32)).T
            if not isinstance(ground_raw, torch.Tensor):
                ground_raw = torch.from_numpy(ground_raw.astype(np.float32)).T
            layers = layers_raw.unsqueeze(0).to(device)
            gt_win = ground_raw.numpy()
            # eta keeps stochasticity during sampling.
            pred = sampler.sample(model, layers, device, eta=0.3, verbose=False)
            pred_windows.append(pred[0].cpu().numpy())
            gt_windows.append(gt_win)
        
        total_len = (len(pred_windows) - 1) * stride + window_size
        from reconstruct_full import hann_overlap_add, remove_spikes
        pred_full = hann_overlap_add(pred_windows, window_size, stride, total_len) * scale
        pred_full -= pred_full.mean(axis=1, keepdims=True)
        pred_full  = remove_spikes(pred_full)
        all_preds.append(pred_full)
    
    # Ground truth only needs to be reconstructed once.
    total_len = (len(gt_windows) - 1) * stride + window_size
    gt_full   = hann_overlap_add(gt_windows, window_size, stride, total_len) * scale
    gt_full  -= gt_full.mean(axis=1, keepdims=True)
    
    stack    = np.stack(all_preds, axis=0)   # [n, 2, T]
    mean_full = stack.mean(axis=0)            # [2, T]
    std_full  = stack.std(axis=0)             # [2, T]
    
    return mean_full, std_full, gt_full

# Find all windows for the target record.
idx = 0
record_samples = []
for meta in val_ds.target_files:
    fname    = meta['filename'].replace('.npz', '')
    npz_path = os.path.join(val_ds.root_dir, meta['folder'], fname + '.npz')
    try:
        data      = np.load(npz_path, allow_pickle=True)
        total_len = data['labels'].shape[0]
        n_win     = max(1, (total_len - 1024) // 512 + 1)
    except:
        n_win = 1
    if fname == target_fname:
        record_samples = [val_ds.samples[idx + i] 
                         for i in range(n_win) 
                         if idx + i < len(val_ds.samples)]
        break
    idx += n_win

print(f"Found {len(record_samples)} windows for {target_fname}")

# Run ensemble uncertainty quantification.
mean_full, std_full, gt_full = ensemble_uq_full(
    model, sampler, record_samples, DEVICE,
    scale=val_ds.scale, n=N_ENSEMBLE
)

T    = gt_full.shape[1]
fs   = 100.0
time = np.arange(T) / fs

# Detect the key segment automatically.
t_start, t_end = detect_key_segment(gt_full, fs=fs)
i_start = int(t_start * fs)
i_end   = int(t_end   * fs)

# Plot the full time history and key-segment zoom.
C_GT   = '#1A1A2E'
C_PRED = '#D95319'

fig, axs = plt.subplots(2, 2, figsize=(18, 10))
plt.rcParams.update({'font.weight':'bold', 'axes.labelsize':13,
                     'axes.titlesize':13, 'font.size':12})

for ch, d in enumerate(['NS', 'EW']):
    # Full time history
    ax0 = axs[ch, 0]
    ax0.plot(time, gt_full[ch],   color=C_GT,   lw=1.2, label='Ground Truth')
    ax0.plot(time, mean_full[ch], color=C_PRED, lw=1.0, ls='--', label='Pred Mean')
    ax0.fill_between(time,
                     mean_full[ch] - 1.96*std_full[ch],
                     mean_full[ch] + 1.96*std_full[ch],
                     color=C_PRED, alpha=0.2, label='95% CI')
    ax0.axvspan(t_start, t_end, color='#FFF3E0', alpha=0.5)
    ax0.set_title(f'{d}  Full Time History')
    ax0.set_xlabel('Time (s)'); ax0.set_ylabel('Displacement')
    ax0.legend(fontsize=10); ax0.grid(True, alpha=0.3)

    # Key-segment zoom
    ax1 = axs[ch, 1]
    time_z = time[i_start:i_end]
    ax1.plot(time_z, gt_full[ch][i_start:i_end],
             color=C_GT,   lw=1.5, label='Ground Truth')
    ax1.plot(time_z, mean_full[ch][i_start:i_end],
             color=C_PRED, lw=1.3, ls='--', label='Pred Mean')
    ax1.fill_between(time_z,
                     mean_full[ch][i_start:i_end] - 1.96*std_full[ch][i_start:i_end],
                     mean_full[ch][i_start:i_end] + 1.96*std_full[ch][i_start:i_end],
                     color=C_PRED, alpha=0.2, label='95% CI')
    # Annotate the mean uncertainty.
    mean_std = std_full[ch][i_start:i_end].mean()
    ax1.text(0.02, 0.96, f'Mean $\\sigma$ = {mean_std:.4f}',
             transform=ax1.transAxes, fontsize=11, va='top', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    ax1.set_title(f'{d}  Key Segment [{t_start:.1f}s – {t_end:.1f}s]')
    ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Displacement')
    ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)

plt.suptitle(f'Ensemble UQ (N={N_ENSEMBLE}) — {target_fname}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'uq_{target_fname}.png', dpi=150, bbox_inches='tight')
plt.show()
print("UQ figure saved.")

In [ ]:
from scipy.ndimage import uniform_filter1d

layer_names = ['L1-NS', 'L1-EW', 'L2-NS', 'L2-EW', 'L3-NS', 'L3-EW']

# Use the middle key-segment window for sensitivity analysis.
best_sample = record_samples[len(record_samples)//2]
layers_raw = best_sample['layers']

if not isinstance(layers_raw, torch.Tensor):
    layers_raw = torch.from_numpy(layers_raw.astype(np.float32)).T

condition_grad = layers_raw.unsqueeze(0).to(DEVICE).requires_grad_(True)

model.eval()
t_fixed = torch.full((1,), 200, device=DEVICE, dtype=torch.long)
x_noisy = torch.randn(1, 2, 1024, device=DEVICE)

saliency = {}

with torch.enable_grad():
    pred = model(x_noisy, condition_grad, t_fixed)

    for ch, d in enumerate(['NS', 'EW']):
        loss = pred[0, ch, :].abs().mean()
        loss.backward(retain_graph=True)

        grad = condition_grad.grad[0].abs().detach().cpu().numpy()
        grad_norm = grad / (grad.max(axis=1, keepdims=True) + 1e-8)
        grad_smooth = uniform_filter1d(grad_norm, size=80, axis=1)

        saliency[d] = grad_smooth
        condition_grad.grad.zero_()


# Plot each channel on its own baseline.
colors = ['#1A1A2E', '#D95319', '#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
time_w = np.arange(1024) / 100.0
offset = 1.2   # Vertical spacing between curves

fig, axs = plt.subplots(1, 2, figsize=(18, 8))
plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'font.size': 12
})

label_x = -0.35   # Left-side label position

for ax, d in zip(axs, ['NS', 'EW']):
    sal = saliency[d]   # [6, 1024]

    for ch in range(6):
        baseline = ch * offset

        # Filled area
        ax.fill_between(
            time_w,
            baseline,
            baseline + sal[ch],
            color=colors[ch],
            alpha=0.6
        )

        # Curve outline
        ax.plot(
            time_w,
            baseline + sal[ch],
            color=colors[ch],
            lw=1.0,
            alpha=0.9
        )

        # Baseline
        ax.axhline(
            baseline,
            color=colors[ch],
            lw=0.5,
            ls='--',
            alpha=0.4
        )

        # Place all channel labels on the left.
        ax.text(
            label_x,
            baseline + 0.3,
            layer_names[ch],
            ha='right',
            va='center',
            fontsize=11,
            color=colors[ch],
            fontweight='bold'
        )

    # Reserve space for left-side labels.
    ax.set_xlim([label_x - 0.15, time_w[-1]])
    ax.set_ylim([-0.2, 6 * offset])

    ax.set_xlabel('Time (s)')
    ax.set_title(f'{d} Output — Input Sensitivity')
    ax.grid(True, axis='x', alpha=0.25)

    ax.set_yticks([])
    ax.tick_params(axis='y', left=False, labelleft=False)

    # Axis spine settings
    ax.spines['left'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_linewidth(1.2)

    ax.tick_params(axis='x', length=4, width=1.0)


plt.tight_layout()
plt.savefig(
    f'sensitivity_{target_fname}.png',
    dpi=150,
    bbox_inches='tight'
)
plt.show()

print("Sensitivity figure saved.")


In [ ]:
# Compute the correlation between pointwise standard deviation and absolute error.
error_ns = np.abs(mean_full[0] - gt_full[0])
error_ew = np.abs(mean_full[1] - gt_full[1])
std_ns   = std_full[0]
std_ew   = std_full[1]

corr_ns_cal = np.corrcoef(std_ns, error_ns)[0, 1]
corr_ew_cal = np.corrcoef(std_ew, error_ew)[0, 1]

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
plt.rcParams.update({'font.weight':'bold','axes.labelsize':13,
                     'axes.titlesize':13,'font.size':12})

for ax, std, err, corr, d in zip(
        axs,
        [std_ns, std_ew],
        [error_ns, error_ew],
        [corr_ns_cal, corr_ew_cal],
        ['NS', 'EW']):
    ax.scatter(std, err, alpha=0.3, s=8, color='#D95319')
    # Fit a linear trend.
    z = np.polyfit(std, err, 1)
    p = np.poly1d(z)
    x_line = np.linspace(std.min(), std.max(), 100)
    ax.plot(x_line, p(x_line), color='#1A1A2E', lw=1.5, ls='--', label='Linear fit')
    ax.set_xlabel('Predictive Std $\\sigma(t)$')
    ax.set_ylabel('Absolute Error $|\\hat{x}(t) - x(t)|$')
    ax.set_title(f'{d}  Uncertainty vs. Error\nPearson $r$ = {corr:.3f}')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'uncertainty_calibration_{target_fname}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Calibration: CorrNS={corr_ns_cal:.3f}  CorrEW={corr_ew_cal:.3f}")


In [ ]:
# Full validation-set calibration analysis.
from reconstruct_full import hann_overlap_add, remove_spikes

def ensemble_uq_full(model, sampler, record_samples, device,
                     scale, n=10, window_size=1024, stride=512, eta=0.3):
    all_preds = []
    gt_windows = []
    for _ in range(n):
        pred_windows = []
        gt_windows_tmp = []
        for sample in record_samples:
            layers_raw = sample['layers']
            ground_raw = sample['ground']
            if not isinstance(layers_raw, torch.Tensor):
                layers_raw = torch.from_numpy(layers_raw.astype(np.float32)).T
            if not isinstance(ground_raw, torch.Tensor):
                ground_raw = torch.from_numpy(ground_raw.astype(np.float32)).T
            layers = layers_raw.unsqueeze(0).to(device)
            with torch.no_grad():
                pred = sampler.sample(model, layers, device, eta=eta, verbose=False)
            pred_windows.append(pred[0].cpu().numpy())
            gt_windows_tmp.append(ground_raw.numpy())
        total_len = (len(pred_windows) - 1) * stride + window_size
        pred_full = hann_overlap_add(pred_windows, window_size, stride, total_len) * scale
        pred_full -= pred_full.mean(axis=1, keepdims=True)
        pred_full  = remove_spikes(pred_full)
        all_preds.append(pred_full)
        if len(gt_windows) == 0:
            gt_windows = gt_windows_tmp

    total_len = (len(gt_windows) - 1) * stride + window_size
    gt_full   = hann_overlap_add(gt_windows, window_size, stride, total_len) * scale
    gt_full  -= gt_full.mean(axis=1, keepdims=True)

    stack     = np.stack(all_preds, axis=0)
    mean_full = stack.mean(axis=0)
    std_full  = stack.std(axis=0)
    return mean_full, std_full, gt_full

record_map = {}
idx = 0
for meta in val_ds.target_files:
    fname    = meta['filename'].replace('.npz', '')
    npz_path = os.path.join(val_ds.root_dir, meta['folder'], fname + '.npz')
    try:
        data      = np.load(npz_path, allow_pickle=True)
        total_len = data['labels'].shape[0]
        n_win     = max(1, (total_len - 1024) // 512 + 1)
    except:
        n_win = 1
    record_map[fname] = [val_ds.samples[idx + i]
                         for i in range(n_win)
                         if idx + i < len(val_ds.samples)]
    idx += n_win

all_corr_ns, all_corr_ew = [], []

for fname, samples in record_map.items():
    if not samples:
        continue
    print(f"Processing {fname}...")
    mean_f, std_f, gt_f = ensemble_uq_full(
        model, sampler, samples, DEVICE,
        scale=val_ds.scale, n=N_ENSEMBLE, eta=0.3
    )
    gt_std = gt_f.std() + 1e-8
    error_ns = np.abs(mean_f[0] - gt_f[0]) / gt_std
    error_ew = np.abs(mean_f[1] - gt_f[1]) / gt_std
    std_ns   = std_f[0] / gt_std
    std_ew   = std_f[1] / gt_std

    r_ns = np.corrcoef(std_ns, error_ns)[0, 1]
    r_ew = np.corrcoef(std_ew, error_ew)[0, 1]
    all_corr_ns.append(r_ns)
    all_corr_ew.append(r_ew)
    print(f"  CorrNS={r_ns:.3f}  CorrEW={r_ew:.3f}")

print(f"\n{'='*45}")
print(f"Full calibration results ({len(all_corr_ns)} records)")
print(f"CorrNS: {np.mean(all_corr_ns):.3f} ± {np.std(all_corr_ns):.3f}")
print(f"CorrEW: {np.mean(all_corr_ew):.3f} ± {np.std(all_corr_ew):.3f}")
print(f"{'='*45}")